# Offensive Language Detection — Practical Project

This notebook runs the practical part of the project:

1. Load dataset
2. Show aggregate dataset statistics
3. Train classical ML models
4. Evaluate models
5. Compare results
6. Display confusion matrices

**Ethical note:** raw text examples are not printed because the dataset may contain harmful language.


In [ ]:
# Cell 1 — Imports and paths
from pathlib import Path
import json
import pandas as pd

ROOT = Path('..').resolve()
OUTPUTS = ROOT / 'outputs'
METRICS = OUTPUTS / 'metrics'
FIGURES = OUTPUTS / 'figures'

print('Project root:', ROOT)


## 1. Dataset Loading

Recommended dataset: **OLID / OffensEval 2019**.

For quick execution, this notebook uses the Hugging Face TweetEval offensive split by default.

To use local OLID instead, run the command from the README:

```bash
python -m src.train_classical --source local --train_path data/raw/olid-training-v1.0.tsv
```


In [ ]:
# Cell 2 — Load data through project utilities
import sys
sys.path.append(str(ROOT))

from src.data_loader import load_hf_tweet_eval, describe_dataset

splits = load_hf_tweet_eval()
train_df = pd.concat([splits.train, splits.validation], ignore_index=True)
test_df = splits.test

summary = {
    'train': describe_dataset(train_df),
    'test': describe_dataset(test_df),
}

summary


## 2. Label Distribution

We show only aggregate statistics, not raw text examples.


In [ ]:
# Cell 3 — Label distribution
label_names = {0: 'NOT', 1: 'OFF'}
label_distribution = (
    train_df['label']
    .map(label_names)
    .value_counts()
    .rename_axis('label')
    .reset_index(name='count')
)
label_distribution


In [ ]:
# Cell 4 — Plot label distribution
import matplotlib.pyplot as plt

ax = label_distribution.plot(kind='bar', x='label', y='count', legend=False, figsize=(6, 4))
ax.set_title('Training Label Distribution')
ax.set_xlabel('Label')
ax.set_ylabel('Count')
plt.tight_layout()
plt.show()


## 3. Train Classical Models

This runs:

1. TF-IDF + Logistic Regression
2. TF-IDF + Linear SVM


In [ ]:
# Cell 5 — Train classical models
from src.train_classical import train_and_evaluate
import argparse

args = argparse.Namespace(
    source='hf_tweet_eval',
    train_path=None,
    test_path=None,
    text_col=None,
    label_col=None,
    test_size=0.2,
    output_dir=str(OUTPUTS),
)

comparison = train_and_evaluate(args)
comparison


## 4. Model Comparison

The comparison table is saved to:

```text
outputs/metrics/model_comparison.csv
```


In [ ]:
# Cell 6 — Load saved comparison table
comparison_path = METRICS / 'model_comparison.csv'
comparison = pd.read_csv(comparison_path)
comparison


## 5. Confusion Matrices

The training script saves confusion matrix images under:

```text
outputs/figures/
```


In [ ]:
# Cell 7 — Display generated confusion matrices
from IPython.display import Image, display

for image_path in sorted(FIGURES.glob('*confusion_matrix.png')):
    print(image_path.name)
    display(Image(filename=str(image_path)))


## 6. Interpretation Template

Use this structure in the final report:

- Which model achieved the highest Macro F1-score?
- Which model had better recall for the offensive class?
- Did classical models perform well enough compared to the transformer model?
- What are the ethical limitations of automatic offensive-language detection?
